# Week 37

In [ ]:
import pandas as pd
import re
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.util import ngrams
import math
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import FreqDist, ConditionalFreqDist
import torch
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from torch import nn
from tqdm.notebook import tqdm
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from typing import List, Tuple
from transformers import MarianMTModel, MarianTokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from tqdm import tqdm
import pandas as pd

## Load the dataset
splits = {'train': 'train.parquet', 'validation': 'validation.parquet'}
df_train = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["train"])
df_val = pd.read_parquet("hf://datasets/coastalcph/tydi_xor_rc/" + splits["validation"])


### Functions from week 36

In [ ]:
## Remove unwanted characters from the questions

def cleanDf(df):
    pattern = re.compile(r"[?؟,;\/\\\[\]#():]")
    # pattern_context = re.compile(r"[?؟,;\/\\\[\]#():.]")
    df['question'] = df['question'].apply(lambda x: pattern.sub("", x))
    df['context'] = df['context'].apply(lambda x: pattern.sub("", x))
    return df

df_train_clean = cleanDf(df_train)
df_val_clean = cleanDf(df_val)

In [ ]:
langForStat = ['ar','ko','te']

numQuestions = []
totalWordCount = []
distinctWordCount = []
distinctCharCount = []

df_train_clean = cleanDf(df_train)
df_val_clean = cleanDf(df_val)

for lang in langForStat:
    numQuestions_train = df_train_clean[df_train_clean['lang'] == lang].shape[0]
    numQuestions_val = df_val_clean[df_val_clean['lang'] == lang].shape[0]
    numQuestions.append((lang, numQuestions_train, numQuestions_val))
    print(f"Language: {lang}, Train Questions: {numQuestions_train}, Validation Questions: {numQuestions_val}")

    # Compute word and character statistics
    df_train_lang = df_train_clean[df_train_clean['lang'] == lang].copy()
    df_val_lang = df_val_clean[df_val_clean['lang'] == lang]
    df_train_lang['wordcount'] = df_train_lang['question'].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)


    maxId = df_train_lang['wordcount'].idxmax()
    longest_question = df_train_lang.loc[maxId, "question"]
    max_words = df_train_lang.loc[maxId, "wordcount"]

    # print(f"Language: {lang}")
    # print(f"  Longest train question (index {maxId}): {longest_question}")
    # print(f"  Word count: {max_words}")

    totalWordCount_train = df_train_lang['question'].apply(lambda x: len(x.split())).sum()
    totalWordCount_val = df_val_lang['question'].apply(lambda x: len(x.split())).sum()
    totalWordCount.append((lang, totalWordCount_train, totalWordCount_val))
    print(f"Language: {lang}, Train Total Words: {totalWordCount_train}, Validation Total Words: {totalWordCount_val}")


### Week 37

In [ ]:
arabicDf_train = df_train_clean[df_train_clean['lang'] == 'ar'].copy()
teluguDf_train = df_train_clean[df_train_clean['lang'] == 'te'].copy()
koreanDf_train = df_train_clean[df_train_clean['lang'] == 'ko'].copy()

arabicDf_val = df_val_clean[df_val_clean['lang'] == 'ar'].copy()
teluguDf_val = df_val_clean[df_val_clean['lang'] == 'te'].copy()
koreanDf_val = df_val_clean[df_val_clean['lang'] == 'ko'].copy()

In [ ]:
def compute_probability(question, unigram_fd, bigram_fd):
    tokens = nltk.word_tokenize(question)
    bigrams = list(ngrams(tokens, 2))
    prob = 1.0
    for bigram in bigrams:
        bigram_count = bigram_fd[bigram]
        unigram_count = unigram_fd[(bigram[0],)]
        if unigram_count > 0 and bigram_count > 0:
            prob *= bigram_count / unigram_count
        else:
            prob *= 1e-6
    return prob

def compute_logprob(question, unigram_fd, bigram_fd):
    tokens = nltk.word_tokenize(question)
    bigrams_list = list(ngrams(tokens, 2))
    log_prob = 0.0
    for bigram in bigrams_list:
        bigram_count = bigram_fd[bigram]
        unigram_count = unigram_fd[(bigram[0],)]
        if unigram_count > 0 and bigram_count > 0:
            prob = bigram_count / unigram_count
        else:
            prob = 1e-6  # smoothing for unseen bigrams
        log_prob += math.log(prob)
    return log_prob, len(tokens)  # return log_prob and number of tokens


In [ ]:
def build_counts(corpus):
    allUnigrams = []
    allBigrams = []
    allTrigrams = []

    for text in corpus:
        tokens = nltk.word_tokenize(text)
        allUnigrams.extend(tokens)
        allBigrams.extend(list(ngrams(tokens, 2)))
        allTrigrams.extend(list(ngrams(tokens, 3)))

    unigram_fd = FreqDist(allUnigrams)
    bigram_fd = FreqDist(allBigrams)
    trigram_fd = FreqDist(allTrigrams)

    return unigram_fd, bigram_fd, trigram_fd


# build counts
unigram_fd, bigram_fd, trigram_fd = build_counts(df_train_clean['context'])
V = len(unigram_fd)

def conditional_prob_unigram(w, unigram_fd):
    return unigram_fd[w] / sum(unigram_fd.values())

def conditional_prob_bigram(w2, w1, bigram_fd, unigram_fd, k=0.0):
    # Add-k for bigram if you like, or simple MLE
    bi = bigram_fd[(w1,w2)]
    uni = unigram_fd[w1]
    if uni>0:
        return bi/uni
    return 1.0/V

def conditional_prob_trigram(w3, w1, w2, trigram_fd, bigram_fd, V, k=0.0):
    tri = trigram_fd[(w1,w2,w3)]
    bi  = bigram_fd[(w1,w2)]
    if bi>0:
        return (tri + k) / (bi + k*V)
    return 1.0/V

def sentence_logprob_interpolated(sentence, unigram_fd, bigram_fd, trigram_fd,
                                  V, lambdas=(0.1,0.3,0.6), k=0.0):
    # lambdas: (lambda_uni, lambda_bi, lambda_tri) must sum to 1
    lam1, lam2, lam3 = lambdas
    toks = nltk.word_tokenize(sentence)
    trigs = list(ngrams(toks, 3))
    logp = 0.0
    for w1,w2,w3 in trigs:
        p_uni = conditional_prob_unigram(w3, unigram_fd)
        p_bi  = conditional_prob_bigram(w3, w2, bigram_fd, unigram_fd, k)
        p_tri = conditional_prob_trigram(w3, w1, w2, trigram_fd, bigram_fd, V, k)
        p = lam1*p_uni + lam2*p_bi + lam3*p_tri
        logp += math.log(p)
    return logp, len(toks)

### Arabic

In [ ]:
# --- TRAINING: unigram model ---
allUnigrams_ar = []

for q in arabicDf_train['question']:
    tokens = nltk.word_tokenize(q)
    allUnigrams_ar.extend(tokens)

unigram_fd_ar = FreqDist(allUnigrams_ar)
total_tokens_train = sum(unigram_fd_ar.values())
V = len(unigram_fd_ar)  # vocabulary size

# print(f"Most common unigrams: {unigram_fd_en.most_common(10)}")

# --- FUNCTION: log-probability of a sentence under unigram model ---
def question_logprob_unigram(sentence, unigram_fd, total_tokens, V, smoothing=1e-6):
    tokens = nltk.word_tokenize(sentence)
    log_prob = 0.0
    for w in tokens:
        count = unigram_fd[w]
        if count > 0:
            prob = count / total_tokens
        else:
            prob = smoothing  # unseen word → small probability
        log_prob += math.log(prob)
    return log_prob, len(tokens)

# --- VALIDATION: calculate perplexity ---
total_log_prob_ar = 0.0
total_tokens_ar = 0

for q in arabicDf_val['question']:
    logp, n = question_logprob_unigram(q, unigram_fd_ar, total_tokens_train, V)
    total_log_prob_ar += logp
    total_tokens_ar += n

perplexity_uni_ar = math.exp(-total_log_prob_ar / total_tokens_ar)
# print("Validation Perplexity (Unigram) for Arabic:", perplexity_uni_ar)

In [ ]:
# Make a list of all bigrams in arabicDf questions
allBigrams_ar = []
allUnigrams_ar = []
for q in arabicDf_train['question']:
    tokens = nltk.word_tokenize(q)
    bigrams = list(ngrams(tokens, 2))
    unigrams = list(ngrams(tokens, 1))
    allBigrams_ar.extend(bigrams)
    allUnigrams_ar.extend(unigrams)

unigram_fd_ar = FreqDist(allUnigrams_ar)
bigram_fd_ar = FreqDist(allBigrams_ar)
# print(unigram_fd_ar.most_common(10))
# print(bigram_fd_ar.most_common(10))
cfdist_ar = ConditionalFreqDist((bigram[0], bigram) for bigram in allBigrams_ar)
# print(cfdist_ar['ما'].most_common(10))

# Length of vocabulary
V_ar = len(unigram_fd_ar)
print(f"Vocabulary size (Arabic): {V_ar}")
# Number of tokens in training set
total_tokens_train = sum(unigram_fd_ar.values())
print(f"Total tokens in training set (Arabic): {total_tokens_train}")

In [ ]:
arabicDf_val['question_prob'] = arabicDf_val['question'].apply(lambda q: compute_probability(q, unigram_fd_ar, bigram_fd_ar))

In [ ]:
# Calculate total log probability and total tokens over validation set
total_log_prob_ar = 0.0
total_tokens_ar  = 0
for q in arabicDf_val['question']:
    logp, n = compute_logprob(q, unigram_fd_ar, bigram_fd_ar)
    total_log_prob_ar += logp
    total_tokens_ar += n

# Perplexity
perplexity_bi_ar = math.exp(-total_log_prob_ar / total_tokens_ar)
# print("Validation Perplexity for Arabic:", perplexity_bi_ar)

In [ ]:
### Trigram model - Arabic

# --- TRAINING: trigram + bigram + unigram ---
allTrigrams_ar = []
allBigrams_ar = []
allUnigrams_ar = []

for q in arabicDf_train['question']:
    tokens = nltk.word_tokenize(q)
    allUnigrams_ar.extend(tokens)
    allBigrams_ar.extend(list(ngrams(tokens, 2)))
    allTrigrams_ar.extend(list(ngrams(tokens, 3)))

unigram_fd_ar = FreqDist(allUnigrams_ar)
bigram_fd_ar = FreqDist(allBigrams_ar)
trigram_fd_ar = FreqDist(allTrigrams_ar)

# print(f"Most common unigrams: {unigram_fd_ar.most_common(10)}")
# print(f"Most common bigrams: {bigram_fd_ar.most_common(10)}")
# print(f"Most common trigrams: {trigram_fd_ar.most_common(10)}")

# --- FUNCTION: log-probability of a sentence under trigram model ---
def question_logprob(sentence, bigram_fd, trigram_fd, smoothing=1e-6):
    tokens = nltk.word_tokenize(sentence)
    trigrams = list(ngrams(tokens, 3))
    log_prob = 0.0
    for w1, w2, w3 in trigrams:
        trigram_count = trigram_fd[(w1, w2, w3)]
        bigram_count = bigram_fd[(w1, w2)]
        if bigram_count > 0 and trigram_count > 0:
            prob = trigram_count / bigram_count
        else:
            prob = smoothing  # unseen trigram → small probability
        log_prob += math.log(prob)
    return log_prob, len(tokens)

# --- VALIDATION: calculate perplexity ---
total_log_prob_ar = 0.0
total_tokens_ar = 0

for q in arabicDf_val['question']:
    logp, n = question_logprob(q, bigram_fd_ar, trigram_fd_ar)
    total_log_prob_ar += logp
    total_tokens_ar += n

perplexity_tri_ar = math.exp(-total_log_prob_ar / total_tokens_ar)
# print("Validation Perplexity (Trigram) for Arabic:", perplexity_tri_ar)

In [ ]:
# Evaluate perplexity on validation set with interpolation
total_log, total_tokens = 0.0, 0
for s in arabicDf_val['question']:
    lp, n = sentence_logprob_interpolated(s, unigram_fd, bigram_fd, trigram_fd, V,
                                          lambdas=(0.1,0.3,0.6), k=0.1)
    total_log += lp
    total_tokens += n
perplexity_inter_ar = math.exp(-total_log/total_tokens)
# print("Interpolated trigram perplexity for Arabic:", perplexity_inter_ar)

In [ ]:
# Analysis of the different models
print(f"Unigram Perplexity (Arabic): {perplexity_uni_ar}")
print(f"Bigram Perplexity (Arabic): {perplexity_bi_ar}")
print(f"Trigram Perplexity (Arabic): {perplexity_tri_ar}")
print(f"Interpolated Trigram Perplexity (Arabic): {perplexity_inter_ar}")

### Korean

In [ ]:
# --- TRAINING: unigram model ---
allUnigrams_ko = []

for q in koreanDf_train['question']:
    tokens = nltk.word_tokenize(q)
    allUnigrams_ko.extend(tokens)

unigram_fd_ko = FreqDist(allUnigrams_ko)
total_tokens_train = sum(unigram_fd_ko.values())
V = len(unigram_fd_ko)  # vocabulary size

# print(f"Most common unigrams: {unigram_fd_en.most_common(10)}")

# --- FUNCTION: log-probability of a sentence under unigram model ---
def question_logprob_unigram(sentence, unigram_fd, total_tokens, V, smoothing=1e-6):
    tokens = nltk.word_tokenize(sentence)
    log_prob = 0.0
    for w in tokens:
        count = unigram_fd[w]
        if count > 0:
            prob = count / total_tokens
        else:
            prob = smoothing  # unseen word → small probability
        log_prob += math.log(prob)
    return log_prob, len(tokens)

# --- VALIDATION: calculate perplexity ---
total_log_prob_ko = 0.0
total_tokens_ko = 0

for q in koreanDf_val['question']:
    logp, n = question_logprob_unigram(q, unigram_fd_ko, total_tokens_train, V)
    total_log_prob_ko += logp
    total_tokens_ko += n

perplexity_uni_ko = math.exp(-total_log_prob_ko / total_tokens_ko)
# print("Validation Perplexity (Unigram) for Korean:", perplexity_uni_ko)

In [ ]:
# Make a list of all bigrams in arabicDf questions
allBigrams_ko = []
allUnigrams_ko = []
for q in koreanDf_train['question']:
    tokens = nltk.word_tokenize(q)
    bigrams = list(ngrams(tokens, 2))
    unigrams = list(ngrams(tokens, 1))
    allBigrams_ko.extend(bigrams)
    allUnigrams_ko.extend(unigrams)

unigram_fd_ko = FreqDist(allUnigrams_ko)
bigram_fd_ko = FreqDist(allBigrams_ko)
print(unigram_fd_ko.most_common(10))
print(bigram_fd_ko.most_common(10))
cfdist_ko = ConditionalFreqDist((bigram[0], bigram) for bigram in allBigrams_ko)

# Length of vocabulary
V = len(unigram_fd_ko)
print(f"Vocabulary size (Korean): {V}")
# Number of tokens in training set
total_tokens_train = sum(unigram_fd_ko.values())
print(f"Total tokens in training set (Korean): {total_tokens_train}")

In [ ]:
koreanDf_val['question_prob'] = koreanDf_val['question'].apply(lambda q: compute_probability(q, unigram_fd_ko, bigram_fd_ko))

In [ ]:
# Calculate total log probability and total tokens over validation set
total_log_prob_ko = 0.0
total_tokens_ko = 0
for q in koreanDf_val['question']:
    logp, n = compute_logprob(q, unigram_fd_ko, bigram_fd_ko)
    total_log_prob_ko += logp
    total_tokens_ko += n

# Perplexity
perplexity_bi_ko = math.exp(-total_log_prob_ko / total_tokens_ko)
print("Validation Perplexity for Korean:", perplexity_bi_ko)

In [ ]:
### Trigram model - Korean

# --- TRAINING: trigram + bigram + unigram ---
allTrigrams_ko = []
allBigrams_ko = []
allUnigrams_ko = []

for q in koreanDf_train['question']:
    tokens = nltk.word_tokenize(q)
    allUnigrams_ko.extend(tokens)
    allBigrams_ko.extend(list(ngrams(tokens, 2)))
    allTrigrams_ko.extend(list(ngrams(tokens, 3)))

unigram_fd_ko = FreqDist(allUnigrams_ko)
bigram_fd_ko = FreqDist(allBigrams_ko)
trigram_fd_ko = FreqDist(allTrigrams_ko)

# print(f"Most common unigrams: {unigram_fd_ko.most_common(10)}")
# print(f"Most common bigrams: {bigram_fd_ko.most_common(10)}")
# print(f"Most common trigrams: {trigram_fd_ko.most_common(10)}")

# --- FUNCTION: log-probability of a sentence under trigram model ---
def question_logprob(sentence, bigram_fd, trigram_fd, smoothing=1e-6):
    tokens = nltk.word_tokenize(sentence)
    trigrams = list(ngrams(tokens, 3))
    log_prob = 0.0
    for w1, w2, w3 in trigrams:
        trigram_count = trigram_fd[(w1, w2, w3)]
        bigram_count = bigram_fd[(w1, w2)]
        if bigram_count > 0 and trigram_count > 0:
            prob = trigram_count / bigram_count
        else:
            prob = smoothing  # unseen trigram → small probability
        log_prob += math.log(prob)
    return log_prob, len(tokens)

# --- VALIDATION: calculate perplexity ---
total_log_prob_ko = 0.0
total_tokens_ko = 0

for q in koreanDf_val['question']:
    logp, n = question_logprob(q, bigram_fd_ko, trigram_fd_ko)
    total_log_prob_ko += logp
    total_tokens_ko += n

perplexity_tri_ko = math.exp(-total_log_prob_ko / total_tokens_ko)
# print("Validation Perplexity (Trigram) for Korean:", perplexity_tri_ko)

In [ ]:
# Evaluate perplexity on validation set with interpolation
total_log, total_tokens = 0.0, 0
for s in koreanDf_val['question']:
    lp, n = sentence_logprob_interpolated(s, unigram_fd, bigram_fd, trigram_fd, V,
                                          lambdas=(0.1,0.3,0.6), k=0.1)
    total_log += lp
    total_tokens += n
perplexity_inter_ko = math.exp(-total_log/total_tokens)

In [ ]:
# Analysis of the different models
print(f"Unigram Perplexity (Korean): {perplexity_uni_ko}")
print(f"Bigram Perplexity (Korean): {perplexity_bi_ko}")
print(f"Trigram Perplexity (Korean): {perplexity_tri_ko}")
print(f"Interpolated Trigram Perplexity (Korean): {perplexity_inter_ko}")

### Telugu

In [ ]:
# --- TRAINING: unigram model ---
allUnigrams_te = []

for q in teluguDf_train['question']:
    tokens = nltk.word_tokenize(q)
    allUnigrams_te.extend(tokens)

unigram_fd_te = FreqDist(allUnigrams_te)
total_tokens_train = sum(unigram_fd_te.values())
V = len(unigram_fd_te)  # vocabulary size

# print(f"Most common unigrams: {unigram_fd_en.most_common(10)}")

# --- FUNCTION: log-probability of a sentence under unigram model ---
def question_logprob_unigram(sentence, unigram_fd, total_tokens, V, smoothing=1e-6):
    tokens = nltk.word_tokenize(sentence)
    log_prob = 0.0
    for w in tokens:
        count = unigram_fd[w]
        if count > 0:
            prob = count / total_tokens
        else:
            prob = smoothing  # unseen word → small probability
        log_prob += math.log(prob)
    return log_prob, len(tokens)

# --- VALIDATION: calculate perplexity ---
total_log_prob_te = 0.0
total_tokens_te = 0

for q in teluguDf_val['question']:
    logp, n = question_logprob_unigram(q, unigram_fd_te, total_tokens_train, V)
    total_log_prob_te += logp
    total_tokens_te += n

perplexity_uni_te = math.exp(-total_log_prob_te / total_tokens_te)
# print("Validation Perplexity (Unigram) for Telugu:", perplexity_uni_te)

In [ ]:
# Make a list of all bigrams in arabicDf questions
allBigrams_telugu = []
allUnigrams_telugu = []
for q in teluguDf_train['question']:
    tokens = nltk.word_tokenize(q)
    bigrams = list(ngrams(tokens, 2))
    unigrams = list(ngrams(tokens, 1))
    allBigrams_telugu.extend(bigrams)
    allUnigrams_telugu.extend(unigrams)

unigram_fd_te = FreqDist(allUnigrams_telugu)
bigram_fd_te = FreqDist(allBigrams_telugu)
# print(unigram_fd_te.most_common(10))
# print(bigram_fd_te.most_common(10))
cfdist_telugu = ConditionalFreqDist((bigram[0], bigram) for bigram in allBigrams_telugu)

# Length of vocabulary
V_te = len(unigram_fd_te)
print(f"Vocabulary size (Telugu): {V_te}")
# Number of tokens in training set
total_tokens_train = sum(unigram_fd_te.values())
print(f"Total tokens in training set (Telugu): {total_tokens_train}")


In [ ]:
teluguDf_val['question_prob'] = teluguDf_val['question'].apply(lambda q: compute_probability(q, unigram_fd_te, bigram_fd_te))

In [ ]:
# Calculate total log probability and total tokens over validation set
total_log_prob_te = 0.0
total_tokens_te = 0
for q in teluguDf_val['question']:
    logp, n = compute_logprob(q, unigram_fd_te, bigram_fd_te)
    total_log_prob_te += logp
    total_tokens_te += n

# Perplexity
perplexity_bi_te = math.exp(-total_log_prob_te / total_tokens_te)
# print("Validation Perplexity for Telugu:", perplexity_bi_te)

In [ ]:
### Trigram model - Telugu

# --- TRAINING: trigram + bigram + unigram ---
allTrigrams_te = []
allBigrams_te = []
allUnigrams_te = []

for q in teluguDf_train['question']:
    tokens = nltk.word_tokenize(q)
    allUnigrams_te.extend(tokens)
    allBigrams_te.extend(list(ngrams(tokens, 2)))
    allTrigrams_te.extend(list(ngrams(tokens, 3)))

unigram_fd_te = FreqDist(allUnigrams_te)
bigram_fd_te = FreqDist(allBigrams_te)
trigram_fd_te = FreqDist(allTrigrams_te)

# print(f"Most common unigrams: {unigram_fd_te.most_common(10)}")
# print(f"Most common bigrams: {bigram_fd_te.most_common(10)}")
# print(f"Most common trigrams: {trigram_fd_te.most_common(10)}")

# --- FUNCTION: log-probability of a sentence under trigram model ---
def question_logprob(sentence, bigram_fd, trigram_fd, smoothing=1e-6):
    tokens = nltk.word_tokenize(sentence)
    trigrams = list(ngrams(tokens, 3))
    log_prob = 0.0
    for w1, w2, w3 in trigrams:
        trigram_count = trigram_fd[(w1, w2, w3)]
        bigram_count = bigram_fd[(w1, w2)]
        if bigram_count > 0 and trigram_count > 0:
            prob = trigram_count / bigram_count
        else:
            prob = smoothing  # unseen trigram → small probability
        log_prob += math.log(prob)
    return log_prob, len(tokens)

# --- VALIDATION: calculate perplexity ---
total_log_prob_te = 0.0
total_tokens_te = 0

for q in teluguDf_val['question']:
    logp, n = question_logprob(q, bigram_fd_te, trigram_fd_te)
    total_log_prob_te += logp
    total_tokens_te += n

perplexity_tri_te = math.exp(-total_log_prob_te / total_tokens_te)
# print("Validation Perplexity (Trigram) for Telugu:", perplexity_tri_te)

In [ ]:
# Evaluate perplexity on validation set with interpolation
total_log, total_tokens = 0.0, 0
for s in teluguDf_val['question']:
    lp, n = sentence_logprob_interpolated(s, unigram_fd, bigram_fd, trigram_fd, V,
                                          lambdas=(0.1,0.3,0.6), k=0.1)
    total_log += lp
    total_tokens += n
perplexity_inter_te = math.exp(-total_log/total_tokens)
# print("Interpolated trigram perplexity for Telugu:", perplexity_inter_te)

In [ ]:
# Analysis of the different models
print(f"Unigram Perplexity (Telugu): {perplexity_uni_te}")
print(f"Bigram Perplexity (Telugu): {perplexity_bi_te}")
print(f"Trigram Perplexity (Telugu): {perplexity_tri_te}")
print(f"Interpolated Trigram Perplexity (Telugu): {perplexity_inter_te}")

### English

In [ ]:
# --- TRAINING: unigram model ---
allUnigrams_en = []

for q in df_train['context']:
    tokens = nltk.word_tokenize(q)
    allUnigrams_en.extend(tokens)

unigram_fd_en = FreqDist(allUnigrams_en)
total_tokens_train = sum(unigram_fd_en.values())
V = len(unigram_fd_en)  # vocabulary size

# print(f"Most common unigrams: {unigram_fd_en.most_common(10)}")

# --- FUNCTION: log-probability of a sentence under unigram model ---
def question_logprob_unigram(sentence, unigram_fd, total_tokens, V, smoothing=1e-6):
    tokens = nltk.word_tokenize(sentence)
    log_prob = 0.0
    for w in tokens:
        count = unigram_fd[w]
        if count > 0:
            prob = count / total_tokens
        else:
            prob = smoothing  # unseen word → small probability
        log_prob += math.log(prob)
    return log_prob, len(tokens)

# --- VALIDATION: calculate perplexity ---
total_log_prob_en = 0.0
total_tokens_en = 0

for q in df_val['context']:
    logp, n = question_logprob_unigram(q, unigram_fd_en, total_tokens_train, V)
    total_log_prob_en += logp
    total_tokens_en += n

perplexity_uni_en = math.exp(-total_log_prob_en / total_tokens_en)
# print("Validation Perplexity (Unigram) for English:", perplexity_en)

In [ ]:
# Make a list of all bigrams in context
allBigrams_en = []
allUnigrams_en = []
for q in df_train_clean['context']:
    tokens = nltk.word_tokenize(q)
    bigrams = list(ngrams(tokens, 2))
    unigrams = list(ngrams(tokens, 1))
    allBigrams_en.extend(bigrams)
    allUnigrams_en.extend(unigrams)

unigram_fd_en = FreqDist(allUnigrams_en)
bigram_fd_en = FreqDist(allBigrams_en)
# print(unigram_fd_en.most_common(10))
# print(bigram_fd_en.most_common(10))
cfdist_en = ConditionalFreqDist((bigram[0], bigram) for bigram in allBigrams_en)

# Length of vocabulary
V_en = len(unigram_fd_en)
print(f"Vocabulary size (English): {V_en}")
# Number of tokens in training set
total_tokens_train = sum(unigram_fd_en.values())
print(f"Total tokens in training set (English): {total_tokens_train}")


In [ ]:
df_val['context_prob'] = df_val['context'].apply(lambda q: compute_probability(q, unigram_fd_en, bigram_fd_en))

In [ ]:
# Calculate total log probability and total tokens over validation set
total_log_prob_en = 0.0
total_tokens_en = 0
for q in df_val['context']:
    logp, n = compute_logprob(q, unigram_fd_en, bigram_fd_en)
    total_log_prob_en += logp
    total_tokens_en += n

# Perplexity
perplexity_bi_en = math.exp(-total_log_prob_en / total_tokens_en)
# print("Validation Perplexity (Bigram) for English:", perplexity_bi_en)

In [ ]:
### Trigram model - English

# --- TRAINING: trigram + bigram + unigram ---
allTrigrams_en = []
allBigrams_en = []
allUnigrams_en = []

for q in df_train['context']:
    tokens = nltk.word_tokenize(q)
    allUnigrams_en.extend(tokens)
    allBigrams_en.extend(list(ngrams(tokens, 2)))
    allTrigrams_en.extend(list(ngrams(tokens, 3)))

unigram_fd_en = FreqDist(allUnigrams_en)
bigram_fd_en = FreqDist(allBigrams_en)
trigram_fd_en = FreqDist(allTrigrams_en)

# print(f"Most common unigrams: {unigram_fd_en.most_common(10)}")
# print(f"Most common bigrams: {bigram_fd_en.most_common(10)}")
# print(f"Most common trigrams: {trigram_fd_en.most_common(10)}")

# --- FUNCTION: log-probability of a sentence under trigram model ---
def question_logprob(sentence, bigram_fd, trigram_fd, smoothing=1e-6):
    tokens = nltk.word_tokenize(sentence)
    trigrams = list(ngrams(tokens, 3))
    log_prob = 0.0
    for w1, w2, w3 in trigrams:
        trigram_count = trigram_fd[(w1, w2, w3)]
        bigram_count = bigram_fd[(w1, w2)]
        if bigram_count > 0 and trigram_count > 0:
            prob = trigram_count / bigram_count
        else:
            prob = smoothing  # unseen trigram → small probability
        log_prob += math.log(prob)
    return log_prob, len(tokens)

# --- VALIDATION: calculate perplexity ---
total_log_prob_en = 0.0
total_tokens_en = 0

for q in df_val['context']:
    logp, n = question_logprob(q, bigram_fd_en, trigram_fd_en)
    total_log_prob_en += logp
    total_tokens_en += n

perplexity_tri_en = math.exp(-total_log_prob_en / total_tokens_en)
# print("Validation Perplexity (Trigram) for English:", perplexity_tri_en)

In [ ]:
# Evaluate perplexity on validation set with interpolation
total_log, total_tokens = 0.0, 0
for s in df_val['context']:
    lp, n = sentence_logprob_interpolated(s, unigram_fd, bigram_fd, trigram_fd, V,
                                          lambdas=(0.1,0.3,0.6), k=0.1)
    total_log += lp
    total_tokens += n
perplexity_inter_en = math.exp(-total_log/total_tokens)
# print("Interpolated trigram perplexity for English:", perplexity_inter_en)

In [ ]:
# Analysis of the different models
print(f"Unigram Perplexity (English): {perplexity_uni_en}")
print(f"Bigram Perplexity (English): {perplexity_bi_en}")
print(f"Trigram Perplexity (English): {perplexity_tri_en}")
print(f"Interpolated Trigram Perplexity (English): {perplexity_inter_en}")

In [ ]:
print("Arabic vocabulary size:", len(unigram_fd_ar))
print("Korean vocabulary size:", len(unigram_fd_ko))
print("Telugu vocabulary size:", len(unigram_fd_te))
print("English vocabulary size:", len(unigram_fd_en))


print("Arabic total words:", sum(unigram_fd_ar.values()))
print("Korean total words:", sum(unigram_fd_ko.values()))
print("Telugu total words:", sum(unigram_fd_te.values()))
print("English total words:", sum(unigram_fd_en.values()))


# Compute percentage of words in validation set that are also in training set for each language
for lang in langForStat + ['en']:
    if lang == 'en':
        # English: use context column
        train_texts = df_train_clean['context'].dropna().astype(str)
        val_texts = df_val_clean['context'].dropna().astype(str)
    else:
        train_texts = df_train_clean[df_train_clean['lang'] == lang]['question'].dropna().astype(str)
        val_texts = df_val_clean[df_val_clean['lang'] == lang]['question'].dropna().astype(str)

    train_vocab = set()
    for text in train_texts:
        train_vocab.update(text.split())

    val_words = []
    for text in val_texts:
        val_words.extend(text.split())
    if len(val_words) == 0:
        percent = 0.0
    else:
        in_train = sum(1 for w in val_words if w in train_vocab)
        percent = 100 * in_train / len(val_words)
    print(f"{lang}: {percent:.2f}% of validation words are in training vocabulary")